<a href="https://colab.research.google.com/github/Miyagui1972/Document_Management_Program/blob/main/Copia_de_14015_G03_MARCO_DONOSO_EJERCICIO_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

# Mantenimiento predictivo en una planta de mecanizado
## Ejercicio 1 · Primera entrega del proyecto integrador: introducción

**ACI7777 - NRC 14015 - Análisis de datos** · Universidad de Las Américas<br>
Fase 1 de la Metodología Basada en Investigación (MBI): planteamiento del problema y formulación de la pregunta

</div>

| | |
|---|---|
| **Integrantes** | Dilan Gustavo Quispe Mamani<br>Freddy Guillermo Muñoz Collao<br>José Vergara Veas<br>Marco Antonio Donoso Donoso<br>Octavio Guillermo Cabrera Recabarren |
| **Docente** | Felipe Fabián Baesler Abufarde |
| **Sección / NRC** | 14015 |
| **Fecha de entrega** | 03-11-2026 |
| **Entorno** | Google Colab · Python 3 · pandas |

## Contenido

1. Contexto y problema organizacional
2. Dataset utilizado
3. Justificación del dataset
4. Pregunta de investigación
5. Carga del dataset con pandas
6. Descripción inicial: dimensiones, tipos de variables y primeras filas
7. Conclusión
8. Referencias

## 1. Contexto y problema organizacional

Para este proyecto asumimos el rol de analistas de datos en **una planta de mecanizado que fabrica componentes para la industria minera e industrial**. Su operación depende de máquinas que trabajan por turnos y en las que una detención inesperada no solo para esa máquina: retrasa pedidos, obliga a horas extra para recuperar producción y, en el caso de clientes mineros, puede gatillar multas contractuales por incumplimiento de plazos.

Hoy la planta trabaja con **mantenimiento correctivo y preventivo por calendario**. El correctivo llega tarde, cuando la falla ya ocurrió. El preventivo por calendario llega a ciegas: cambia herramientas o detiene equipos según una fecha, aunque la máquina esté en buen estado, o no lo hace a tiempo cuando la exigencia del turno fue mayor a la habitual. En ambos casos se pierde disponibilidad.

La oportunidad está en lo que la máquina ya registra mientras opera: temperaturas, velocidad de giro, torque y desgaste de la herramienta. Si esas condiciones muestran un comportamiento distinto antes de una falla, la planta podría pasar a un **mantenimiento basado en condición**, interviniendo cuando los datos lo justifican. Ese es el problema que guía todo nuestro proyecto.

## 2. Dataset utilizado

| Elemento | Detalle |
|---|---|
| **Nombre** | AI4I 2020 Predictive Maintenance Dataset |
| **Fuente** | UCI Machine Learning Repository (repositorio académico público) |
| **Enlace directo** | https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset |
| **Autor** | Stephan Matzka, HTW Berlin (2020) |
| **Licencia** | Creative Commons Attribution 4.0 (CC BY 4.0) |
| **Contenido** | 10.000 registros de operación de una máquina de fresado, con condiciones de proceso y un indicador de falla |
| **Naturaleza de los datos** | Sintéticos, construidos por el autor para reproducir el comportamiento de datos reales de mantenimiento predictivo en la industria (Matzka, 2020a) |

Que los datos sean sintéticos no le quita valor al ejercicio, pero sí acota lo que podemos afirmar: las conclusiones describirán patrones de este conjunto de datos y no de una planta específica. Lo declaramos desde ahora para tenerlo presente en todas las etapas del proyecto.

## 3. Justificación del dataset y del problema organizacional

**Correspondencia temática.** El dataset registra exactamente el fenómeno que nos preocupa: máquinas de mecanizado operando bajo distintas condiciones y, para cada registro, si terminó o no en falla. No tuvimos que forzar el dataset para que calzara con nuestro caso: sus variables son las mismas que se miden en una máquina real.

**Variables vinculadas con el problema.** Cada variable tiene una lectura operacional directa:

| Variable del dataset | Qué representa en la planta | Por qué importa para el problema |
|---|---|---|
| `Type` (L, M, H) | Calidad del producto en fabricación: baja, media o alta | Distintas exigencias de proceso pueden desgastar la máquina de forma distinta |
| `Air temperature [K]` | Temperatura ambiente del área de trabajo | Condiciona la disipación de calor |
| `Process temperature [K]` | Temperatura del proceso de mecanizado | Un diferencial bajo con el ambiente dificulta enfriar la máquina |
| `Rotational speed [rpm]` | Velocidad de giro del husillo | Junto con el torque, define la potencia exigida |
| `Torque [Nm]` | Esfuerzo de giro | Sobreesfuerzo mecánico y consumo de potencia |
| `Tool wear [min]` | Minutos de uso acumulado de la herramienta | Es el equivalente a las horas de uso que hoy la planta controla por calendario |
| `Machine failure` | 1 si hubo falla, 0 si no | Es la variable que queremos entender y anticipar |
| `TWF`, `HDF`, `PWF`, `OSF`, `RNF` | Modo de falla: desgaste de herramienta, disipación de calor, potencia, sobreesfuerzo y aleatoria | Explican *cómo* falló; sirven para interpretar, no para predecir (Matzka, 2020b) |

**Utilidad de los datos para analizar el problema.** Con estas variables podemos recorrer el ciclo completo del curso: describir en qué condiciones opera normalmente la máquina (análisis descriptivo), comparar las condiciones de los registros con y sin falla (análisis diagnóstico), construir un modelo que estime la probabilidad de falla (aprendizaje supervisado) y agrupar regímenes de operación parecidos para definir criterios de inspección diferenciados (aprendizaje no supervisado). En términos de negocio, cada falla anticipada es una detención no planificada que se convierte en una intervención programada.

## 4. Pregunta de investigación

**Problema de investigación.** Las fallas no planificadas en las máquinas de mecanizado reducen la disponibilidad de la planta, y hoy no sabemos qué condiciones de operación las anteceden. Siguiendo a Hernández Sampieri et al. (2014), partimos de este problema para llegar a una pregunta delimitada en variables, datos y alcance.

**Pregunta de investigación.** ¿Qué condiciones de operación se asocian a la ocurrencia de fallas en máquinas de mecanizado?

**Pregunta analítica (la que guiará el proyecto):**

> ¿En qué medida el desgaste de la herramienta, el torque, la velocidad de rotación, las temperaturas del aire y del proceso y el tipo de producto se asocian con la ocurrencia de una falla de máquina, y con qué capacidad permiten anticiparla, en los 10.000 registros del dataset AI4I 2020?

| Criterio de la rúbrica | Cómo lo cumple la pregunta |
|---|---|
| **Medible** | Todas las variables son numéricas o categóricas registradas; la falla es binaria (0/1) y su asociación y predicción se miden con estadísticos y métricas concretas |
| **Específica** | Delimita seis variables explicativas, una variable de resultado y un conjunto de datos definido |
| **Relevante para la organización** | Responde directamente a la necesidad de pasar de mantenimiento por calendario a mantenimiento basado en condición |
| **Respondible con el dataset** | Todas las variables mencionadas existen en el archivo, como se verifica en la sección 5 |

**Hipótesis de trabajo.** Esperamos que los registros con falla presenten mayor desgaste acumulado de la herramienta y combinaciones extremas de torque y velocidad (muy alto torque a baja velocidad, o a la inversa), más que diferencias marcadas de temperatura por sí solas. Esta hipótesis se pondrá a prueba en el análisis exploratorio y en el modelamiento de las próximas entregas.

## 5. Carga del dataset con pandas

Para que el notebook se pueda ejecutar de principio a fin sin pasos manuales, el código primero busca el archivo original descargado desde UCI (`ai4i2020.csv`) en la carpeta de trabajo. Si no lo encuentra, lo lee desde una copia pública del mismo archivo alojada en GitHub. En ambos casos se verifica que la estructura sea la esperada antes de continuar.

In [ ]:
# Librerías necesarias para esta entrega
import os
import pandas as pd

# Mostrar todas las columnas al imprimir tablas
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

In [ ]:
# Ubicaciones del dataset:
# 1) archivo original descargado desde UCI y subido a Colab (opción preferida)
# 2) copia pública del mismo archivo en GitHub (respaldo para que el notebook siempre corra)
ruta_local = "ai4i2020.csv"
url_respaldo = ("https://raw.githubusercontent.com/nandhinig05/"
                "ai4i2020_prediction_machinefailure/HEAD/ai4i2020.csv")

if os.path.exists(ruta_local):
    df_maquinas = pd.read_csv(ruta_local)
    origen = "archivo local descargado desde UCI"
else:
    df_maquinas = pd.read_csv(url_respaldo)
    origen = "copia pública en GitHub"

print(f"Dataset cargado desde: {origen}")

Dataset cargado desde: copia pública en GitHub


In [ ]:
# Verificación: el dataset contiene todas las variables que exige la pregunta analítica
variables_pregunta = {
    "Type", "Air temperature [K]", "Process temperature [K]",
    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]", "Machine failure",
}
variables_faltantes = variables_pregunta - set(df_maquinas.columns)

if variables_faltantes:
    print("Faltan variables necesarias:", variables_faltantes)
else:
    print("El dataset contiene las 7 variables que requiere la pregunta analítica.")

El dataset contiene las 7 variables que requiere la pregunta analítica.


## 6. Descripción inicial del dataset

### 6.1 Dimensiones (`.shape`)

In [ ]:
# Cantidad de registros (filas) y variables (columnas)
n_registros, n_variables = df_maquinas.shape
print(f"Dimensiones: {df_maquinas.shape}")
print(f"El dataset tiene {n_registros:,} registros y {n_variables} variables.".replace(",", "."))

Dimensiones: (10000, 14)
El dataset tiene 10.000 registros y 14 variables.


**Interpretación.** El conjunto tiene **10.000 registros y 14 variables**. Cada fila es una observación de operación de la máquina con su resultado (falla o no falla). Es un volumen adecuado para el curso: suficiente para separar datos de entrenamiento y prueba en la Unidad 3 sin que Colab tenga problemas de procesamiento.

### 6.2 Tipos de variables (`.dtypes`)

In [ ]:
# Tipo de dato con que pandas leyó cada columna
print(df_maquinas.dtypes)

UDI                          int64
Product ID                  object
Type                        object
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object


**Interpretación.** pandas reconoce **12 columnas numéricas** (enteras y decimales) y **2 de texto** (`Product ID` y `Type`). Leído desde el problema, las variables se agrupan así:

| Grupo | Variables | Tipo estadístico | Rol en el proyecto |
|---|---|---|---|
| Identificadores | `UDI`, `Product ID` | Identificador | Se excluyen del análisis: no describen la operación |
| Condiciones de proceso | `Air temperature [K]`, `Process temperature [K]`, `Torque [Nm]` | Cuantitativas continuas | Variables explicativas |
| Condiciones de proceso | `Rotational speed [rpm]`, `Tool wear [min]` | Cuantitativas continuas (se registran como enteros) | Variables explicativas |
| Tipo de producto | `Type` (L, M, H) | Cualitativa ordinal | Variable explicativa; deberá codificarse respetando su orden |
| Resultado | `Machine failure` | Cualitativa binaria (0/1) | **Variable objetivo** |
| Modo de falla | `TWF`, `HDF`, `PWF`, `OSF`, `RNF` | Cualitativas binarias | Solo para interpretar; si se usaran para predecir, el modelo «haría trampa», porque solo se conocen después de la falla |

Los tipos asignados por pandas son coherentes con el contenido de cada columna, así que en esta etapa no se requieren conversiones. La única transformación prevista es codificar `Type` en la etapa de preprocesamiento.

### 6.3 Primeras cinco filas (`.head()`)

In [ ]:
# Primeras cinco observaciones para revisar formato y valores
df_maquinas.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


**Interpretación.** Las primeras filas confirman que los datos están limpios en su forma: temperaturas en kelvin cercanas a 298 K (≈ 25 °C) para el aire y 308 K (≈ 35 °C) para el proceso, velocidades entre 1.400 y 1.550 rpm, torques de 40 a 50 Nm aproximadamente y herramientas que recién comienzan su vida útil (0 a 9 minutos de uso). Ninguno de estos cinco registros terminó en falla, lo que anticipa algo que conviene medir de inmediato: qué tan frecuentes son las fallas en el total del dataset.

### 6.4 Revisión complementaria: completitud y frecuencia de fallas

Esta revisión no es exigida por el enunciado, pero responde la pregunta que dejó abierta el `.head()` y adelanta dos decisiones clave para las próximas entregas.

In [ ]:
# Valores faltantes y registros duplicados
print(f"Valores nulos en todo el dataset: {df_maquinas.isnull().sum().sum()}")
print(f"Registros duplicados: {df_maquinas.duplicated().sum()}")

# Frecuencia de la variable objetivo
conteo_fallas = df_maquinas["Machine failure"].value_counts()
porcentaje_fallas = df_maquinas["Machine failure"].value_counts(normalize=True) * 100
resumen_fallas = pd.DataFrame({"registros": conteo_fallas, "porcentaje": porcentaje_fallas.round(2)})
resumen_fallas.index = resumen_fallas.index.map({0: "Sin falla (0)", 1: "Con falla (1)"})
resumen_fallas

Valores nulos en todo el dataset: 0
Registros duplicados: 0


,registros,porcentaje
Machine failure,,
Sin falla (0),9661,96.61
Con falla (1),339,3.39


In [ ]:
# Tasa de falla según tipo de producto
tasa_por_tipo = (df_maquinas.groupby("Type")["Machine failure"].mean() * 100).round(2)
print("Tasa de falla (%) por tipo de producto:")
print(tasa_por_tipo.reindex(["L", "M", "H"]))

Tasa de falla (%) por tipo de producto:
Type
L    3.92
M    2.77
H    2.09
Name: Machine failure, dtype: float64


**Interpretación.** No hay valores nulos ni filas duplicadas. El dato que más nos llamó la atención es que **solo 339 de los 10.000 registros (3,39 %) terminan en falla**. Para la operación eso es bueno, pero para el análisis es una advertencia. Un modelo que dijera siempre «no habrá falla» acertaría el 96,6 % de las veces sin detectar ni una sola falla, así que en la Unidad 3 la exactitud (*accuracy*) no podrá ser nuestra métrica principal.

También aparece una primera señal descriptiva: los productos de calidad baja (L) muestran una tasa de falla cercana al 3,9 %, frente a 2,8 % en M y 2,1 % en H. Es solo una diferencia observada; si es relevante o no lo diremos después de analizarla con más rigor.

## 7. Conclusión

El dataset AI4I 2020 **sí permite responder la pregunta analítica**. Contiene las seis condiciones de operación y la variable de falla que la pregunta necesita, con tipos de datos correctos, sin valores faltantes y con un volumen suficiente para las técnicas del curso. Además, su temática coincide con un problema real y costoso para cualquier planta industrial: anticipar fallas para intervenir antes de la detención.

Esta primera revisión deja tres decisiones para las próximas entregas:

1. **Desbalance de clases.** Con 3,39 % de fallas, evaluaremos los modelos con métricas como *recall* y F1 sobre la clase de falla, y no solo con exactitud.
2. **Variables que no se usarán para predecir.** Los identificadores (`UDI`, `Product ID`) se excluyen, y los modos de falla (`TWF` a `RNF`) quedan solo para interpretar los resultados, porque se conocen después de ocurrida la falla.
3. **Alcance de las conclusiones.** Al tratarse de datos sintéticos, los resultados describirán patrones del conjunto analizado y no podrán trasladarse directamente a una planta sin validarlos con sus propios registros.

## 8. Referencias

Hernández Sampieri, R., Fernández Collado, C., & Baptista Lucio, M. del P. (2014). *Metodología de la investigación* (6.ª ed.). McGraw-Hill Interamericana.

Matzka, S. (2020a). *AI4I 2020 predictive maintenance dataset* [Conjunto de datos]. UCI Machine Learning Repository. https://doi.org/10.24432/C5HS5C

Matzka, S. (2020b). Explainable artificial intelligence for predictive maintenance applications. En *2020 Third International Conference on Artificial Intelligence for Industries (AI4I)* (pp. 69–74). IEEE. https://doi.org/10.1109/AI4I49448.2020.00023
